## Extract vital signs and nursing reports for the first 48hr ICU stay

### Prerequisite: build the following derived tables according to official repo https://github.com/MIT-LCP/mimic-code/tree/main/mimic-iii
echo_data;
height_first_day;
pivoted_vital;
weight_durations;
pivoted_fio2;
pivoted_gcs;
pivoted_lab;

### the tables below can be built using our code
first_stay;
capillary;
ph;



#### packages
transformers==4.10.2

In [1]:
#libraries
import numpy as np
import pandas as pd
import psycopg2 #used to connect to our local MIMIC-III database
import collections
# import getpass
from datetime import datetime
import os,sys,re
import pickle
import csv
import math
#import seaborn as sns
# import random
from datetime import timedelta
from pathlib import Path
import importlib
import bisect
import glob
import statistics


from notebook.services.config import ConfigManager
cm = ConfigManager()
cm.update('livereveal', {
        'width': 1024,
        'height': 768,
        'scroll': True,
})

#%load_ext autotime

{'width': 1024, 'height': 768, 'scroll': True}

In [2]:
## try to connect MIMICIII

dbname = 'mimiciii'
password = 'YourPassword'
user = 'postgres'
conn = psycopg2.connect(dbname=dbname, password=password,user=user,port=5433)
cur=conn.cursor()

### Common SQL functions to PSQL functions

In [ ]:
'''
round(max(3),2) ---> round(max(3::numeric),2)

DATETIME_SUB(intime, INTERVAL 12 HOUR) ---> intime - '12 hours'::interval

DATETIME_ADD(intime, INTERVAL 12 HOUR) ---> intime + '12 hours'::interval

FORMAT_DATE('%Y-%m-%d', chartdate) ---> make_date(extract(year from chartdate)::int, extract(month from chartdate)::int, extract(day from chartdate)::int)

REGEXP_CONTAINS(c.value, '[^0-9\\.]') ---> c.value like '%.%'
'''

### Capillary refill and pH (SKIP IF DONE)

In [ ]:
'''
in chartevents for capillary refill
itemid = 3348, value = 'Brisk' or 'Delayed'
itemid = 223951 (capillary right) or 224308 (capillary left), value = 'Normal <3 Seconds' or 'Abnormal >3 Seconds'

'''

# build capillary refill table if not exsists

query = '''
create table capillary as 
select subject_id, icustay_id, charttime, case when
(value like 'Brisk' or value like 'Normal <3 Seconds') 
then 1 else 0 end as capillary_normal,
case when (value like 'Delayed' or value like 'Abnormal >3 Seconds')
then 1 else 0 end as capillary_abnormal 
from chartevents where itemid = 3348 or itemid = 223951

'''
cur.execute(query)


# build ph table if not exsists


query = '''

create table pH as

select lab.subject_id, icu.icustay_id, lab.charttime, lab.valuenum as pH from labevents lab join icustays icu on
lab.subject_id = icu.subject_id where lab.itemid = 50820 and lab.valuenum is not null and icu.intime <= lab.charttime and
icu.outtime >= lab.charttime
'''
cur.execute(query)

## For documentation: PSQL 12 code for echo_data table

In [ ]:
'''
create table echo_data as (
select t2.ROW_ID, t2.subject_id,t2.hadm_id, t2.Indication, t2.Height, t2.Weight, t2.BSA, 
t2.BP, t2.BPSys, t2.BPDias, t2.HR, t2.Status, t2.Test, t2.Doppler,t2.Contrast,t2.TechnicalQuality,
t3.chartdate, make_timestamp
  (extract(year from t3.chartdate)::int,
  extract(month from t3.chartdate)::int,
  extract(day from t3.chartdate)::int,
  left(t2.sub_str,2)::int,
  right(t2.sub_str,2)::int,
  0
   ) AS charttime from  (
select * from (
select t1.ROW_ID, t1.subject_id,t1.hadm_id, substring(t1.text, 'Date/Time: .+? at ([0-9]+:[0-9]{2})') as sub_str,
substring(t1.text, 'Indication: (.*?)\n') as Indication
 , cast(substring(t1.text, 'Height: \(in\) ([0-9]+)') as numeric) as Height
  , cast(substring(t1.text, 'Weight \(lb\): ([0-9]+)\n') as numeric) as Weight
  , cast(substring(t1.text, 'BSA \(m2\): ([0-9]+) m2\n') as numeric) as BSA -- ends in 'm2'
  , substring(t1.text, 'BP \(mm Hg\): (.+)\n') as BP -- Sys/Dias
  , cast(substring(t1.text, 'BP \(mm Hg\): ([0-9]+)/[0-9]+?\n') as numeric) as BPSys -- first part of fraction
  , cast(substring(t1.text, 'BP \(mm Hg\): [0-9]+/([0-9]+?)\n') as numeric) as BPDias -- second part of fraction
  , cast(substring(t1.text, 'HR \(bpm\): ([0-9]+?)\n') as numeric) as HR

  , substring(t1.text, 'Status: (.*?)\n') as Status
  , substring(t1.text, 'Test: (.*?)\n') as Test
  , substring(t1.text, 'Doppler: (.*?)\n') as Doppler
  , substring(t1.text, 'Contrast: (.*?)\n') as Contrast
  , substring(t1.text, 'Technical Quality: (.*?)\n') as TechnicalQuality
  
from noteevents t1 where substring(t1.text, 'Date/Time: .+? at ([0-9]+:[0-9]{2})') is not null and t1.category = 'Echo' and t1.hadm_id is not null)t5) t2 join 
(select subject_id,hadm_id,chartdate from noteevents where category = 'Echo') t3
on t2.hadm_id = t3.hadm_id

)



'''

### Select cohort. Only first stay.

#### First create a table for first ICU stay. Skip if the table exists.

In [14]:
query = '''
DROP TABLE IF EXISTS first_stay;
create table first_stay as 
select rk.subject_id, rk.icustay_id, rk.intime,rk.outtime, rk.los from( SELECT subject_id, icustay_id, intime,outtime, los, 
RANK() OVER (PARTITION BY subject_id ORDER BY intime asc) as RN
FROM icustays) rk where rk.rn=1 and rk.los>=2

'''
cur.execute(query)


# los = length of stay in days, first 48hr considered


'''
dynamic features
capillary refill rate; [capillary]
diastolic blood pressure; [pivoted_vital]
fraction inspired oxygen; [pivoted_fio2]
the eye opening, [pivoted_gcs]
motor response, [pivoted_gcs]
verbal response, [pivoted_gcs]
total value of the Glasgow Coma Scale; [pivoted_gcs]
glucose; [pivoted_vital]
heart rate; [pivoted_vital]
mean blood pressure; [pivoted_vital]
oxygen saturation; [pivoted_vital]
respiratory rate; [pivoted_vital]
systolic blood pressure; [pivoted_vital]
temperature; [pivoted_vital]
pH [pH]


static features
age
sex
race
weight
height


'''

In [3]:
from nltk import sent_tokenize, word_tokenize
import nltk
#nltk.download('punkt_tab') #download vocab if necessarr
#nltk.download('punkt') 
import re
import torch


SECTION_TITLES = re.compile(
    r'(NEURO|CV|GU/GI|GU|GI|RESP|ENDO|PLAN|SOCIAL|CARDIAC|ENDO|CVS|LYTES|SKIN|O|P'
    r'|G&D|FEN|F&N|DEV|PARENTS|BILI|SEPSIS|DEVE|PARENTING|FEN O|G&D O|INC|ACTIVITY'
    r'|ID|[** **]|#1|#2|#3|#4|#5|#6|#7|#8|#9|#10):|#1|#2|#3|#4|#5|#6|#7|#8'
    r'|#9|#10|1.\)|2.\)|3.\)|4.\)|5.\)|6.\)|7.\)|8.\)|9.\)|10.\)',re.I | re.M)


max_length = 128


def split_heading(text):
    """Split the report into sections"""
    start = 0
    for matcher in SECTION_TITLES.finditer(text):
        # add last
        end = matcher.start()
        if end != start:
            section = text[start:end].strip()
            if section:
                yield section

        # add title
        start = end
        end = matcher.end()
        if end != start:
            section = text[start:end].strip()
            if section:
                yield section

        start = end

    # add last piece
    end = len(text)
    if start < end:
        section = text[start:end].strip()
        if section:
            yield section


            

def preprocess_clinicalbert(text):
    text = text.replace('\n', ' ')
    text = text.replace('\r', ' ')
    text = text.strip().lower()
    text = re.sub('\\[(.*?)\\]', '', text)  # remove de-identified brackets
    text = re.sub('[0-9]+\.', '', text)  # remove 1.2. since the segmenter segments based on this
    text = re.sub('m\.d\.', 'md', text)
    text = re.sub('admission date:', '', text)
    text = re.sub('discharge date:', '', text)
    text = re.sub('--|__|==', '', text)
    return text
    
    

def clean_text(text):
    """
    Clean text
    """

    # Replace [**Patterns**] with spaces.
    text = re.sub(r'\[\*\*.*?\*\*\]', pattern_repl, text)
    # Replace `_` with spaces.
    text = re.sub(r'_', ' ', text)

    

    start = 0
    #new_text = ''
    #if start > 0:
    #    new_text += ' ' * start
    #new_text = text[start:]
    end = len(text)
    new_text = text

    # make sure the new text has the same length of old text.
    if len(text) - end > 0:
        new_text += ' ' * (len(text) - end)
        
    return new_text


def preprocess_mimic(text):
    """
    Preprocess reports in MIMIC-III.
    1. remove [**Patterns**] and signature
    2. split the report into sections
    3. tokenize sentences and words
    4. lowercase
    """
    for sec in split_heading(clean_text(text)):
        for sent in sent_tokenize(sec):
            text = ' '.join(word_tokenize(sent))
            yield text.lower()
            

def getSentences(t):
    return list(preprocess_mimic(t))




def pattern_repl(matchobj):
    """
    Return a replacement string to be used for match object
    """
    return ' '.rjust(len(matchobj.group(0)))




def txt2embd(text, tokenizer, bert):
    # text: str
    
    # split report into sentences str ----> [str1, str2,...]
    sentences = getSentences(text)
    if len(sentences) == 0:
        return None
    encoded_sent = tokenizer.encode_plus(
                    text=sentences,                      # Preprocess sentence
                    add_special_tokens=True,        # Add [CLS] and [SEP]
                    max_length=max_length,             # Max length to truncate/pad
                    pad_to_max_length=True,         # Pad sentence to max length
                    #return_tensors='pt',           # Return PyTorch tensor
                    return_attention_mask=True,     # Return attention mask
                    truncation=True
                    )
    Textarr = torch.tensor([encoded_sent.get('input_ids')])
    Attnarr = torch.tensor([encoded_sent.get('attention_mask')])
    txtemb = bert.bert(Textarr, Attnarr)
    emb = txtemb[0][0] # (128, 768)
    # Attnarr[0].tolist() (128)
    return emb.tolist(), Attnarr[0].tolist()
    
def load_BERT(freeze=True):
    # freeze: load BERT without backpropagation
    path = '../clinicalbert_cache/ClinicalBERT.pickle'
    clinical_bert = Base_ClinicalBERT(path, freeze=freeze)
    path = '../clinicalbert_cache/tokenizer.pickle'
    tokenizer = load_tokenizer(path)
    return clinical_bert, tokenizer


class Base_ClinicalBERT(torch.nn.Module):
    def __init__(self, path, freeze=False):
        super(Base_ClinicalBERT, self).__init__()
        self.model_name = 'emilyalsentzer/Bio_ClinicalBERT'
        if os.path.isfile(path):
            with open(path, 'rb') as file:
                self.bert = pickle.load(file) 
        else:
            from transformers import BertModel
            self.bert = BertModel.from_pretrained(self.model_name,
                                                  return_dict=False)
            with open(path, 'wb') as file:
                pickle.dump(self.bert, file)
        if freeze:
            for p in self.bert.parameters():
                p.requires_grad = False
            
def load_tokenizer(path):
    if os.path.isfile(path):
        with open(path, 'rb') as file:
            tokenizer = pickle.load(file)
    else:
        from transformers import BertTokenizer
        tokenizer = BertTokenizer.from_pretrained('emilyalsentzer/Bio_ClinicalBERT',
                                                  do_lower_case=True)
        with open(path, 'wb') as file:
            pickle.dump(tokenizer, file)
    return tokenizer





def get_preprocessed_text(notes):
    # notes: {id:{tim1:note1,...}}
    exclude = []
    for stay_id in notes:
        idv = notes[stay_id]
        exclude_time = []
        times = list(notes[stay_id].keys())
        for time in times:
            sentences = preprocess_clinicalbert(idv[time][0])
            if len(sentences) == 0:
                exclude_time.append(time)
            else:
                notes[stay_id][time] = sentences
        if len(exclude_time) > 0:
            del notes[stay_id][time]
        if len(notes[stay_id].keys()) == 0:
            exclude.append(stay_id)
    if len(exclude) > 0:
        for stay_id in exclude:
            del notes[stay_id]
    return notes




def get_split_sentences(notes):
    # notes: {id:{tim1:note1,...}}
    exclude = []
    for stay_id in notes:
        idv = notes[stay_id]
        exclude_time = []
        times = list(notes[stay_id].keys())
        for time in times:
            sentences = getSentences(idv[time][0])
            if len(sentences) == 0:
                exclude_time.append(time)
            else:
                notes[stay_id][time] = sentences
        if len(exclude_time) > 0:
            del notes[stay_id][time]
        if len(notes[stay_id].keys()) == 0:
            exclude.append(stay_id)
    if len(exclude) > 0:
        for stay_id in exclude:
            del notes[stay_id]
    return notes


def get_emb(notes):
    # notes: {id:{tim1:note1,...}}
    bert, tokenizer = load_BERT()
    exclude = []
    for stay_id in notes:
        times = list(notes[stay_id].keys())
        idv = notes[stay_id]
        exclude_time = []
        for time in times:
            emb_attn = txt2embd(idv[time][0], tokenizer, bert)
            if emb_attn is None:
                exclude_time.append(time)
            else:
                notes[stay_id][time] = emb_attn # (emb, attn_mask)
        if len(exclude_time) > 0:
            for time in exclude_time:
                del notes[stay_id][time]
        if len(notes[stay_id].keys()) == 0:
            exclude.append(stay_id)
    if len(exclude) > 0:
        for stay_id in exclude:
            del notes[stay_id]
    return notes



def median_impute(df, columns):
    # df has column icustay_id
    for col in columns:
        median = df[col].dropna(how='any').median() # global median
        stay_median = {}
        for stay,val in zip(df['icustay_id'], df[col]):
            if stay not in stay_median:
                stay_median[stay] = []
            if not math.isnan(float(val)):
                stay_median[stay].append(float(val))
        stay_median1 = {}
        for i in stay_median:
            vals = stay_median[i]
            if len(vals) == 0:
                stay_median1[i] = median # global median
            else:
                stay_median1[i] = np.median(vals) # local median
        new_values = []
        for stay,val in zip(df['icustay_id'], df[col]):
            if math.isnan(float(val)):
                new_values.append(stay_median1[stay])
            else:
                new_values.append(float(val))
        df[col] = new_values
    return df

def datetime_to_sec(df,time_col='charttime'):
    time_in_sec = []
    date_format = '%Y-%m-%d %H:%M:%S'
    for i in df[time_col]:
        time_in_sec.append(datetime.strptime(str(i),date_format).timestamp())
    df[time_col] = time_in_sec
    return df


def feature_dict(df, feature_start=2, stay=0, time=1):
    result = {}
    n = len(df)
    for i in range(n):
        row = list(df.iloc[i])
        stay_id = row[stay]
        charttime = row[time]
        features = row[feature_start:]
        if stay_id not in result:
            result[stay_id] = {}
        result[stay_id][charttime] = features
    return result # {icustay_id:{charttime1:[features],charttime2:[features],...}, ...} 


def label_dict(df, label_start=1, stay=0):
    result = {}
    n = len(df)
    for i in range(n):
        row = list(df.iloc[i])
        stay_id = row[stay]
        features = row[label_start:]
        result[stay_id] = features
    return result



def down_sample(dict1, interval=1800):
    # interval: sec
    # down sample the df if there too many records
    result = {}
    for stay_id in dict1:
        result[stay_id] = {}
        times = sorted(list(dict1[stay_id].keys()))
        current = times[0]
        total = len(times)
        last_idx = -1
        while bisect.bisect_left(times, current) < total:
            idx = bisect.bisect_left(times, current)
            current += interval
            if last_idx == idx:
                continue
            else:
                last_idx = idx
            timestamp = times[idx]
            result[stay_id][timestamp] = dict1[stay_id][timestamp]
    return result


def get_median(data_dict):
    data = []
    median = []
    for stay_id in data_dict:
        for time in data_dict[stay_id].keys():
            data.append(data_dict[stay_id][time])
    col = len(data[0])
    data = np.array(data)
    for i in range(col):
        median.append(statistics.median(data[:,i]))
    return median
    

def merge_dict(df1, df2):
    # df1: dict, df2: dict
    # iterate timestamps of df1
    result = {}
    global_median = get_median(df2)
    for stay_id in df1:
        if stay_id not in df2:
            df2[stay_id] = {}
            starttime = min(list(df1[stay_id].keys()))
            df2[stay_id][starttime] = global_median
        result[stay_id] = {}
        times1 = sorted(list(df1[stay_id].keys()))
        times2 = sorted(list(df2[stay_id].keys()))
        total = len(times2)
        for timestamp in times1:
            idx = bisect.bisect_left(times2, timestamp)
            if idx == total:
                timestamp2 = times2[-1]
            else:
                timestamp2 = times2[idx]
            result[stay_id][timestamp] = df1[stay_id][timestamp]
            result[stay_id][timestamp].extend(df2[stay_id][timestamp2])
    del df2
    return result

### Extract static features

In [4]:

#path = './static_features.pickle'
#if os.path.isfile(path):
#    file = open(path,'rb')
#    static_features = pickle.load(file)
#    file.close()
#else:
    query = '''
select demo1.subject_id, demo1.icustay_id, demo1.age, demo1.is_male, demo1.is_female, demo1.is_asian, demo1.is_white,demo1.is_black,
demo1.is_hispanic,demo1.is_other, demo1.weight, ht.height from (
select demo.subject_id, demo.icustay_id, demo.age, demo.is_male, demo.is_female, demo.is_asian, demo.is_white,demo.is_black,
demo.is_hispanic,demo.is_other, wt.weight from ( 
select first.subject_id, first.icustay_id, extract(year from age(first.intime, pt1.dob)) as age, pt1.is_male, pt1.is_female,
pt1.is_asian, pt1.is_white, pt1.is_black, pt1.is_hispanic, pt1.is_other from first_stay first join (
select pt.subject_id, pt.dob, 
race.is_asian, race.is_white, race.is_black, race.is_hispanic, race.is_other,
case when gender like 'M' then 1 else 0 end as is_male,
      case when gender like 'F' then 1 else 0 end as is_female from patients pt join (
select subject_id, case when 
(ethnicity in ('ASIAN - VIETNAMESE', 'ASIAN', 'ASIAN - THAI', 'ASIAN - ASIAN INDIAN', 'MIDDLE EASTERN', 'ASIAN - KOREAN', 'AMERICAN INDIAN/ALASKA NATIVE','AMERICAN INDIAN/ALASKA NATIVE FEDERALLY RECOGNIZED TRIBE',
'ASIAN - CHINESE', 'ASIAN - FILIPINO', 'ASIAN - OTHER', 'ASIAN - CAMBODIAN', 'ASIAN - JAPANESE')) then 1
                 else 0
                 end as is_asian,
                 case 
                 when (ethnicity in ('WHITE - OTHER EUROPEAN', 'WHITE - EASTERN EUROPEAN', 'WHITE','WHITE - BRAZILIAN',
                 'WHITE - RUSSIAN')) then 1 else 0
                 end as is_white,
                 case
                 when  (ethnicity in ('BLACK/HAITIAN', 'BLACK/AFRICAN', 'BLACK/AFRICAN AMERICAN', 'BLACK/CAPE VERDEAN'
                 )) then 1 else 0
                 end as is_black,
                 case
                 when  (ethnicity in ('PORTUGUESE', 'SOUTH AMERICAN', 'HISPANIC/LATINO - SALVADORAN', 'HISPANIC/LATINO - COLOMBIAN',
                 'HISPANIC/LATINO - CENTRAL AMERICAN (OTHER)', 'CARIBBEAN ISLAND', 'HISPANIC/LATINO - CUBAN', 'HISPANIC/LATINO - PUERTO RICAN',
                 'HISPANIC/LATINO - HONDURAN', 'HISPANIC OR LATINO', 'HISPANIC/LATINO - GUATEMALAN', 'HISPANIC/LATINO - MEXICAN',
                 'HISPANIC/LATINO - DOMINICAN')) then 1 else 0
                 end as is_hispanic,
                 case
                 when  (ethnicity in ('OTHER', 'UNKNOWN/NOT SPECIFIED', 'NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER',
                 'MULTI RACE ETHNICITY', 'PATIENT DECLINED TO ANSWER', 'UNABLE TO OBTAIN' )) then 1 else 0
                 end as is_other
                 from admissions
                 ) race on race.subject_id = pt.subject_id) pt1 on first.subject_id = pt1.subject_id) demo
                 join weight_first_day wt on demo.icustay_id = wt.icustay_id where wt.weight is not null and demo.age is not null
                 ) demo1 join height_first_day ht on demo1.icustay_id = ht.icustay_id where ht.height is not null
'''
#    static_features = pd.read_sql_query(query,conn)
#    filehandler = open(path,"wb")
#    pickle.dump(static_features,filehandler)
#    filehandler.close()
#static_features

IndentationError: unexpected indent (<ipython-input-4-0441e260f6ec>, line 7)

## Extract height

In [5]:
query = '''

select icustay_id, height from height_first_day where icustay_id in (select icustay_id from first_stay) and height is not null
'''
height = label_dict(pd.read_sql_query(query,conn))


## Extract dynamic features

In [6]:
path = './dynamic_features.pickle'
if os.path.isfile(path):
    file = open(path,'rb')
    dynamic_features = pickle.load(file)
    file.close()
else:   
    
    query ='''
select icustay_id, charttime, heartrate, sysbp, diasbp, meanbp, resprate, tempc, spo2, glucose from pivoted_vital where
icustay_id in (select icustay_id from first_stay)

'''
    pivoted_vital = pd.read_sql_query(query,conn)
    pivoted_vital = median_impute(pivoted_vital, ['heartrate', 'sysbp', 'diasbp', 'meanbp', 'resprate', 'tempc', 'spo2', 
                                                  'glucose'])
    pivoted_vital = down_sample(feature_dict(datetime_to_sec(pivoted_vital)))
    dynamic_features = pivoted_vital
    del pivoted_vital
    
    
    query = '''
    
    select icustay_id, charttime, aniongap, albumin, bands, bicarbonate, bilirubin, creatinine, chloride, glucose, hematocrit,
    hemoglobin, lactate, platelet, potassium, ptt, inr, pt, sodium, bun, wbc 
    from pivoted_lab where icustay_id in (select icustay_id from first_stay)
    '''
    pivoted_lab = pd.read_sql_query(query,conn)
    pivoted_lab = feature_dict(datetime_to_sec(median_impute(pivoted_lab, ['aniongap', 'albumin', 'bands', 'bicarbonate', 'bilirubin', 
                                             'creatinine', 'chloride', 'glucose', 'hematocrit', 'hemoglobin',
                                             'lactate', 'platelet', 'potassium', 'ptt', 'inr',
                                             'pt', 'sodium', 'bun', 'wbc'])))
    dynamic_features = merge_dict(dynamic_features, pivoted_lab)
    
    
    query ='''
select icustay_id, charttime, capillary_normal, capillary_abnormal from capillary where
icustay_id in (select icustay_id from first_stay) and capillary_normal is not null and capillary_abnormal is not null
'''
    capillary = pd.read_sql_query(query,conn)
    capillary = feature_dict(datetime_to_sec(capillary))
    dynamic_features = merge_dict(dynamic_features, capillary)
    
    query = '''
    
    select icustay_id, starttime as charttime, weight from weight_durations where icustay_id in (select icustay_id 
    from first_stay)
    '''
    weight = pd.read_sql_query(query,conn)
    weight = feature_dict(datetime_to_sec(weight))
    dynamic_features = merge_dict(dynamic_features, weight)
    
    
    query ='''
select icustay_id, charttime, fio2 from pivoted_fio2 where
icustay_id in (select icustay_id from first_stay) and fio2 is not null

'''
    pivoted_fio2 = pd.read_sql_query(query,conn)
    pivoted_fio2 = feature_dict(datetime_to_sec(pivoted_fio2))
    dynamic_features = merge_dict(dynamic_features, pivoted_fio2)
    
    query ='''
select icustay_id, charttime, gcs, gcsmotor, gcsverbal, gcseyes from pivoted_gcs where
icustay_id in (select icustay_id from first_stay)

'''
    gcs = pd.read_sql_query(query,conn)
    gcs = median_impute(gcs, ['gcs', 'gcsmotor', 'gcsverbal', 'gcseyes'])
    gcs = feature_dict(datetime_to_sec(gcs))
    dynamic_features = merge_dict(dynamic_features, gcs)
    
    query ='''
select icustay_id, charttime, ph from ph where
icustay_id in (select icustay_id from first_stay) and ph is not null

'''
    ph = pd.read_sql_query(query,conn)
    ph = feature_dict(datetime_to_sec(ph))
    dynamic_features = merge_dict(dynamic_features, ph)
    
    
    
    #### exclude some high missingness patients
    query = '''
    select distinct icustay_id from pivoted_vital where icustay_id in (select icustay_id from first_stay)
    '''
    pivoted_vital_id = list(pd.read_sql_query(query,conn)['icustay_id'])
    
    query = '''
    select distinct icustay_id from pivoted_lab where icustay_id in (select icustay_id from first_stay_sepsis)
    '''
    pivoted_lab_id = list(pd.read_sql_query(query,conn)['icustay_id'])
    
    query = '''
    select distinct icustay_id from capillary where icustay_id in (select icustay_id from first_stay)
    '''
    capillary_id = list(pd.read_sql_query(query,conn)['icustay_id'])
    query = '''
    select distinct icustay_id from weight_durations where icustay_id in (select icustay_id from first_stay)
    '''
    weight_id = list(pd.read_sql_query(query,conn)['icustay_id'])
    query = '''
    select distinct icustay_id from pivoted_fio2 where icustay_id in (select icustay_id from first_stay)
    '''
    pivoted_fio2_id = list(pd.read_sql_query(query,conn)['icustay_id'])
    query = '''
    select distinct icustay_id from pivoted_gcs where icustay_id in (select icustay_id from first_stay)
    '''
    pivoted_gcs_id = list(pd.read_sql_query(query,conn)['icustay_id'])
    query = '''
    select distinct icustay_id from ph where icustay_id in (select icustay_id from first_stay)
    '''
    ph_id = list(pd.read_sql_query(query,conn)['icustay_id'])
    
    exclude_id = []
    min_num_missing = 2 # minimum acceptable
    for icustay_id in pivoted_vital_id:
        missing = 0
        if icustay_id not in pivoted_lab_id:
            missing += 1
        if icustay_id not in capillary_id:
            missing += 1
        if icustay_id not in weight_id:
            missing += 1
        if icustay_id not in pivoted_fio2_id:
            missing += 1
        if icustay_id not in pivoted_gcs_id:
            missing += 1
        if icustay_id not in ph_id:
            missing += 1
        if missing > min_num_missing:
            exclude_id.append(icustay_id)
            
    for icustay_id in exclude_id:
        del dynamic_features[icustay_id]
    file = open(path,'wb')
    pickle.dump(dynamic_features,file)
    file.close()



In [10]:
len(dynamic_features)

10547

## Merge static and dynamic features

In [7]:
#path = './time_series.pickle'
#if os.path.isfile(path):
#    file = open(path,'rb')
#    time_series = pickle.load(file)
#    file.close()
#else:
#    time_series = {}
#    num_row = len(static_features)
#    for idx in range(num_row):
#        row = list(static_features.iloc[idx,1:])
#        icustay_id = row[0]
#        if icustay_id in dynamic_features:
#            time_series[icustay_id] = {}
#            time_series[icustay_id]['static'] = row[1:]
#            time_series[icustay_id]['dynamic'] = dynamic_features[icustay_id]
#        else:
#            continue
#    file = open(path,'wb')
#    pickle.dump(time_series,file)
#    file.close()
#    del static_features
#    del dynamic_features

## Merge dynamic features and height

In [7]:
temp = []
for stay_id in height:
    temp.append(height[stay_id][0])
height_median = statistics.median(temp)
for stay_id in dynamic_features:
    if stay_id in height:
        height_value = height[stay_id][0]
    else:
        height_value = height_median
    for timestamp in dynamic_features[stay_id]:
        dynamic_features[stay_id][timestamp].append(height_value)
        

## Extract nursing reports

In [8]:
path = './notes.pickle'
if os.path.isfile(path):
    file = open(path,'rb')
    notes = pickle.load(file)
    file.close()
else:
    
    query = '''

select first.icustay_id, note.charttime, note.text from noteevents note join first_stay first on 
first.subject_id = note.subject_id where note.charttime >= first.intime and note.charttime <= first.outtime and
note.category = 'Nursing/other' and note.description = 'Report' and 
note.cgid in (select cgid from public.caregivers where label = 'RN') and length(note.text)>100 and 
note.text not like '%Addendum%' and note.text not like '%addendum%' and note.text not like '%ADDENDUM%' 
and note.text not like '%Management Note%' and note.text not like '%management note%' and note.text not like '%Management note%'
and note.text not like '%MANAGEMENT NOTE%' and note.text not like '%Admission Note%' and note.text not like '%Admission note%'
and note.text not like '%admission note%' and note.text not like '%ADMISSION NOTE%'
'''

    #notes = get_split_sentences(feature_dict(datetime_to_sec(pd.read_sql_query(query,conn))))
    notes = get_preprocessed_text(feature_dict(datetime_to_sec(pd.read_sql_query(query,conn))))
    file = open(path,'wb')
    pickle.dump(notes,file)
    file.close()
    

In [8]:
import pickle

path = './notes.pickle'
file = open(path, 'rb')
notes = pickle.load(file)
for id in notes:
    for time in notes[id]:
        print(notes[id][time])
        break
    break

micu npn 7p-7a see admission hx/fhp for pmh/hpi.  events of shift:  arrived from  hospital @  intubated, sedated and paralyzed. placed on mso4, versed and doxacurium gtts upon arrival to unit. ogt, ett confirmed by cxr; a-line and tlc placed overnight (still awaiting confirmation by cxr). ct scan of chest unremarkable per preliminary .  review of systems:  neuro: doxa @ 015 mg/kg/hr, increased when spontaneous movement seen on smaller dose. mso4@ 8 mg/hr, versed @ 8 mg/hr, please keep in mind that pt. has a fairly high tolerance for mso4 given his home regimen of oxycontin and percocet. no spontaneous movement noted after doxa increased. bolused with mso4 for perceived pain (sbp increase to 180's).  resp: pt. remains intubated and vented on a/c 750 x 12 fio2 50% peep  abg will be drawn on these settings. lungs are coarse t/o, rhonchii heard scattered at times, insp. wheezes in r lung. we have not been suctioning pt. in order to preserve the integrity of the clot on the pt.'s carina. if

## Extract labels

In [9]:

query = '''

select first.icustay_id, case when (first.intime <= pt.dod_hosp and first.outtime >= pt.dod_hosp) 
then 1 else 0 end as mortality from first_stay first join patients pt on first.subject_id = pt.subject_id
'''

labels = label_dict(pd.read_sql_query(query,conn))



In [5]:
labels

{211552: [0],
 228232: [0],
 220597: [1],
 288409: [0],
 232669: [0],
 263738: [0],
 277042: [0],
 217847: [0],
 203487: [0],
 244882: [0],
 254478: [1],
 295037: [0],
 282039: [0],
 248910: [0],
 249426: [0],
 261027: [0],
 225852: [0],
 291554: [0],
 224440: [0],
 252348: [0],
 216609: [0],
 232593: [0],
 244776: [0],
 283361: [0],
 294232: [0],
 224214: [0],
 211832: [0],
 239612: [0],
 284305: [0],
 290076: [0],
 254066: [0],
 209562: [0],
 277633: [0],
 249786: [0],
 291001: [0],
 242398: [0],
 211943: [0],
 288376: [0],
 216929: [0],
 251343: [0],
 221100: [0],
 233111: [0],
 245390: [0],
 212246: [0],
 225366: [0],
 294980: [0],
 252051: [1],
 282073: [0],
 254245: [0],
 232514: [0],
 255660: [0],
 282343: [0],
 296852: [0],
 211157: [0],
 267893: [0],
 223465: [0],
 272335: [0],
 219090: [0],
 294530: [0],
 234668: [1],
 293406: [0],
 269520: [0],
 226841: [0],
 260172: [0],
 252947: [0],
 227964: [0],
 224026: [0],
 201239: [0],
 263211: [0],
 279643: [0],
 260696: [0],
 23505

## Merge time series, notes, and labels. First 48hr is considered.

In [31]:
data = {}
duration = 48 * 3600
for stay_id in dynamic_features:
    if stay_id not in notes or stay_id not in labels:
        continue
    ts_times  = sorted(list(dynamic_features[stay_id].keys()))
    note_times = sorted(list(notes[stay_id].keys()))
    note_times1 = sorted(list(notes[stay_id].keys()))
    start = ts_times[0]
    end = start + duration
    ts_times = ts_times[:bisect.bisect_right(ts_times,end)]
    ts_times1 = ts_times[:bisect.bisect_right(ts_times,end)]
    start_idx = bisect.bisect_left(note_times,start)
    end_idx = bisect.bisect_right(note_times,end)
    if start_idx == len(note_times):
        continue
    if start_idx == end_idx:
        if start_idx == 0:
            continue
        else:
            note_times = note_times[start_idx:end_idx+1]
    else:
        note_times = note_times[start_idx:end_idx]
    if (note_times[0] - ts_times[0]) / 3600 > 48:
        print('note_times1',note_times1)
        print('ts_times1',ts_times1)
        break
    data[stay_id] = {}
    data[stay_id]['dynamic'] = {}
    for timestamp in ts_times:
        data[stay_id]['dynamic'][timestamp] = dynamic_features[stay_id][timestamp]
    data[stay_id]['notes'] = {}
    for timestamp in note_times:
        data[stay_id]['notes'][timestamp] = notes[stay_id][timestamp]
    data[stay_id]['label'] = labels[stay_id]
path = './mortality_data.pickle'
file = open(path,'wb')
pickle.dump(data,file)
file.close()
    

In [32]:
len(data)

10973